# 02_ex_agreement_topic — profile + AI draft + human approval


In [ ]:
%run 00_env_config
%run 01_data_agreement_template


In [ ]:
from fabricops_kit import (
    profile_dataframe,
    prepare_business_context_profile_input,
    suggest_column_business_contexts,
    extract_column_business_context_suggestions,
    capture_column_business_context,
    COLUMN_BUSINESS_CONTEXT_FROM_WIDGET,
    prepare_dq_profile_input,
    attach_rule_metadata_keys,
    suggest_dq_rules_with_fabric_ai,
    extract_candidate_rules_from_responses,
    review_dq_rules,
    prepare_governance_profile_input,
    suggest_personal_identifier_classifications,
    extract_personal_identifier_suggestions,
    review_column_governance_context,
    COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET,
    write_column_business_context,
    write_column_governance_context,
    get_path,
)
from fabricops_kit.data_quality import build_dq_rules_metadata_df
import fabricops_kit.data_quality as data_quality


In [ ]:
SOURCE_TABLE = "REPLACE_SOURCE_TABLE"
df_source = spark.table(SOURCE_TABLE)
profile_rows = profile_dataframe(df_source)


In [ ]:
prepared_context_input = prepare_business_context_profile_input(profile_rows.collect(), table_name=SOURCE_TABLE, table_context=BUSINESS_CONTEXT)
prepared_context_df = spark.createDataFrame(prepared_context_input)
ai_context_response_df = suggest_column_business_contexts(prepared_profile_df=prepared_context_df, prompt_template=CONFIG.ai_prompt_config.business_context_template)
ai_context_suggestions = extract_column_business_context_suggestions(ai_context_response_df)
capture_column_business_context(ai_context_suggestions, environment_name=ENV_NAME, dataset_name=DATASET_NAME, table_name=SOURCE_TABLE)
# COLUMN_BUSINESS_CONTEXT_FROM_WIDGET is populated asynchronously by widget callbacks.


In [ ]:
dq_profile_rows = prepare_dq_profile_input(profile_rows.collect(), table_name=SOURCE_TABLE, column_contexts=COLUMN_BUSINESS_CONTEXT_FROM_WIDGET)
dq_profile_df = spark.createDataFrame(dq_profile_rows)
dq_response_df = suggest_dq_rules_with_fabric_ai(prepared_profile_df=dq_profile_df, prompt_template=CONFIG.ai_prompt_config.dq_rule_candidate_template)
candidate_rules = extract_candidate_rules_from_responses(dq_response_df, table_name=SOURCE_TABLE)
candidate_rules = attach_rule_metadata_keys(candidate_rules, environment_name=ENV_NAME, dataset_name=DATASET_NAME, table_name=SOURCE_TABLE)
review_dq_rules(candidate_rules, table_name=SOURCE_TABLE)
approved_rules = data_quality.APPROVED_RULES_FROM_WIDGET
approved_rules_metadata_df = build_dq_rules_metadata_df(spark, approved_rules)


In [ ]:
governance_profile_rows = prepare_governance_profile_input(profile_rows.collect(), table_name=SOURCE_TABLE, column_contexts=COLUMN_BUSINESS_CONTEXT_FROM_WIDGET)
governance_profile_df = spark.createDataFrame(governance_profile_rows)
governance_response_df = suggest_personal_identifier_classifications(prepared_profile_df=governance_profile_df, prompt=CONFIG.ai_prompt_config.governance_personal_identifier_template)
governance_suggestions = extract_personal_identifier_suggestions(governance_response_df)
review_column_governance_context(governance_suggestions, environment_name=ENV_NAME, dataset_name=DATASET_NAME, table_name=SOURCE_TABLE)
approved_governance = COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET


In [ ]:
# Optional metadata persistence
metadata_path = get_path(ENV_NAME, "metadata", config=FABRIC_CONFIG)
# write_column_business_context(spark, COLUMN_BUSINESS_CONTEXT_FROM_WIDGET, metadata_path, table_name="METADATA_COLUMN_CONTEXT")
# approved_rules_metadata_df.write.mode("append").saveAsTable("METADATA_DQ_RULES")
# write_column_governance_context(spark, COLUMN_GOVERNANCE_CONTEXT_FROM_WIDGET, metadata_path, table_name="METADATA_COLUMN_GOVERNANCE")
